# Reproduction — Zeng, Li & Sreenath, CDC 2021

**Paper:** *Enhancing Feasibility and Safety of Nonlinear Model Predictive Control with Discrete-Time Control Barrier Functions*, CDC 2021.

Runs top-to-bottom headless (`jupyter nbconvert --execute`, no `--allow-errors`), asserts its own numbers, saves figures to `figures/` and the summary table to `results_cdc2021.csv` (both gitignored). Scenario constants are the repository's canonical fixture (double integrator, dt = 0.1, obstacle (0.5, 0.5) r_eff = 0.4, goal (1, 1), 20×20 start grid), numerically identical to the gtest/pytest fixtures — see §1.1.

**What this paper adds over ACC 2021**
1. The decay rate becomes a decision variable `omega_k` instead of a fixed `gamma` — relaxing the
   constraint where it would otherwise be infeasible, while keeping `omega_k * gamma <= 1` so safety survives.
2. A *generalised* DCBF applied over a shorter CBF horizon `N_CBF <= N`.
3. An empirical feasibility comparison against the fixed-decay formulation.

**Sections:** §1 formulation + the ωγ ≤ 1 derivation; §2 feasible-region comparison over a γ sweep; §3 realised ω_k trajectories; §4 N_CBF sweep at N = 8; §5 summary → `results_cdc2021.csv`.

In [ ]:
# Imports + determinism. The repo root goes on sys.path so the `codegen`
# package (single source of truth for models + OCP assembly) is importable
# exactly as in the pytest suite and the other notebooks.
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless; CI has no display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    d = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (d / "codegen").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    raise RuntimeError("repo root not found")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

import casadi as ca  # noqa: E402
from acados_template import AcadosOcpSolver  # noqa: E402
from codegen.models import (  # noqa: E402
    MODEL_REGISTRY,
    RNG_SEED,
    barrier_expression,
    discretise,
)
from codegen.generate_mpc_cbf_solver import build_ocp  # noqa: E402

# Fixed seed, printed in the first cell (ground rule: replayable results).
print(f"seed: {RNG_SEED:#x}   (codegen.models.RNG_SEED)")

HERE = REPO_ROOT / "reproduction" / "zeng_cdc2021"
FIGDIR = HERE / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

print(
    "numpy", np.__version__,
    "| casadi", ca.__version__,
    "| matplotlib", matplotlib.__version__,
    "| pandas", pd.__version__,
)

## 1. The relaxed-decay formulation

The repo's fixed-decay DCBF row (ACC 2021, `dcbf_constraint()`) is

$$h(x_{k+1}) - h(x_k) \ge -\gamma\, h(x_k) \quad\Longleftrightarrow\quad h(x_{k+1}) \ge (1-\gamma)\, h(x_k).$$

CDC 2021 replaces the fixed rate $\gamma$ with a **per-step decision variable** $\omega_k \ge 0$:

$$h(x_{k+1}) - h(x_k) \ge -\omega_k \gamma\, h(x_k) \quad\Longleftrightarrow\quad h(x_{k+1}) \ge (1 - \omega_k \gamma)\, h(x_k),$$

with $\omega_k \gamma \le 1$ imposed as the safety condition. **Why $\omega_k \gamma \le 1$ preserves $h \ge 0$** — the entire safety argument:

1. Suppose $h(x_k) \ge 0$ and $\omega_k \gamma \le 1$. Then $(1 - \omega_k \gamma) \ge 0$.
2. Multiplying a non-negative number by $h(x_k) \ge 0$ gives $h(x_{k+1}) \ge (1 - \omega_k \gamma)\, h(x_k) \ge 0$.
3. By induction from a safe start ($h(x_0) \ge 0$, guaranteed by the stage-0 distance row), every state on the closed loop satisfies $h(x_k) \ge 0$.

So the relaxation can *never* trade away safety: whatever $\omega_k$ the solver picks, as long as $\omega_k \gamma \le 1$ the set $\{x : h(x) \ge 0\}$ stays forward-invariant. If the solver needed $\omega_k \gamma > 1$ the multiplier would be negative and the decay bound would allow $h$ to drop below zero — that is the one line the empirical assertion in §3 watches.

**Cost.** $\omega_k$ enters the stage cost as $\omega_{\text{weight}} (\omega_k - 1)^2$ with $\omega_{\text{weight}} = 1000$ (the YAML default): the solver keeps $\omega_k = 1$ (pure ACC 2021 decay) unless relaxing it is genuinely cheaper than the alternatives, and pays quadratically for deviation.

**Bounds.** The repo sets $\omega_k \in [0, 3]$ through the extended input bounds (`omega_max = 3.0` in YAML). The `omega_min = 0` lower bound is what makes the DCBF row a one-sided inequality; the upper bound 3.0 is *not* the safety condition — $\omega_k \gamma \le 1$ is, which the codegen cannot express as a static bound because it couples $\omega$ with the runtime parameter $\gamma$ (checked empirically in §3, and in the pytest suite's A6).

**Implementation.** In the codegen (`generate_mpc_cbf_solver.py`), `relaxed_decay` extends the input vector to $u = [u_{\text{phys}};\, \omega] \in \mathbb{R}^{n_u + n_{\text{obs}}}$, so `nu_total = nu + n_obstacles = 10` for the 8-slot fixture. The DCBF row substitutes $F(x_k, u_k)$ for $x_{k+1}$ — the same substitution as fixed decay — and the cost residual is pre-scaled by $\sqrt{\omega_{\text{weight}}}$ so `W` stays identity on the $\omega$ block (cost $= \omega_{\text{weight}}(\omega-1)^2$).

In [ ]:
# --- Fixture scenario (imported values, not retyped) ----------------------
# Numerically identical to the pytest fixture in
# mpc_cbf_unified/test/test_recursive_feasibility.py (which states the same
# identity against the gtest fixture): dt = 0.1, obstacle at (0.5, 0.5, 0)
# with radius 0.2, ego radius 0.15, safety margin 0.05 -> r_eff = 0.4,
# goal (1, 1, 0, 0), N = 8, 20x20 grid over [-0.5, 1.5]^2 with the obstacle
# interior removed (356 starts), gamma = 0.3, N_OBSTACLES = 8 parameter slots
# (1 real obstacle + 7 far-away dummies, position 1e6 / radius 0).
N_OBSTACLES = 8
DT = 0.1
GAMMA_SWEEP = np.array([0.1, 0.3, 0.7, 1.0])   # §2 sweep (matches §12.4)
GAMMA = 0.3                                     # fixture default
OBSTACLE = {
    "position": np.array([0.5, 0.5, 0.0]),
    "velocity": np.zeros(3),
    "radius": 0.4,          # r_eff = 0.2 obstacle + 0.15 ego + 0.05 margin
    "is_dynamic": False,
}
R_EFF = OBSTACLE["radius"]
X0 = np.array([0.0, 0.0, 0.0, 0.0])
X_GOAL = np.array([1.0, 1.0, 0.0, 0.0])
GOAL_TOL = 0.05
MAX_STEPS = 100
GRID_LO, GRID_HI, N_GRID = -0.5, 1.5, 20

# h(x) from the single source of truth (codegen barrier_expression).
spec2d = MODEL_REGISTRY["double_integrator_2d"]()
_xs = ca.SX.sym("x", spec2d.nx)
_os = ca.SX.sym("o", 7)
_h_fn = ca.Function("h", [_xs, _os], [barrier_expression(spec2d, _xs, _os)])


def barrier(x: np.ndarray, obs7: np.ndarray) -> float:
    """h(x) for one obstacle given its 7-slot parameter block."""
    return float(_h_fn(x, obs7))


def obstacle_slice(obs: dict, stage: int, dt: float = DT) -> np.ndarray:
    """7-slot parameter block for one obstacle at prediction stage `stage`."""
    s = np.zeros(7)
    s[:3] = np.asarray(obs["position"], float)
    if obs.get("is_dynamic", False):
        s[:3] = s[:3] + stage * dt * np.asarray(obs["velocity"], float)
    s[3:6] = np.asarray(obs["velocity"], float)
    s[6] = obs["radius"]
    return s


def parameter_vector(stage: int, obstacles: list[dict], gamma: float,
                     dt: float = DT, n_obstacles: int = N_OBSTACLES) -> np.ndarray:
    """[o_0(7), ..., o_{n-1}(7), gamma]; layout per §5.3 of the design doc."""
    p = np.zeros(7 * n_obstacles + 1)
    for j in range(n_obstacles):
        if j < len(obstacles):
            p[7 * j:7 * j + 7] = obstacle_slice(obstacles[j], stage, dt)
        else:
            p[7 * j:7 * j + 3] = 1.0e6      # far-away dummy, as in the C++ prune pad
    p[7 * n_obstacles] = gamma
    return p


def set_reference(solver, x_ref: np.ndarray, variant: str) -> None:
    """Constant set-point tracked by every stage (inputs penalised at 0)."""
    n_omega = N_OBSTACLES if variant == "relaxed_decay" else 0
    yref = np.concatenate([x_ref, np.zeros(2 + n_omega)])
    N = solver.acados_ocp.dims.N
    for k in range(N):
        solver.set(k, "yref", yref)
    solver.set(N, "yref", x_ref)


# Exact-ZOH step, bit-identical to the solver's discrete dynamics.
F_di = discretise(spec2d, DT, "exact")


def step_double_integrator(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    return np.asarray(F_di(x, u)).flatten()


# --- Solver factory: one build per (model, horizon, variant); gamma is a
# runtime parameter (tail of every stage's p vector), so the whole gamma
# sweep shares one generated solver per variant. Generated code goes under
# results/ (gitignored). The 8-slot fixture (dummies at 1e6) is used here —
# unlike the ACC notebook's paper scenario, where the dummies stalled the
# SQP; the fixture scenario solves cleanly with them (pytest grid: 98.9 %
# feasible at N = 8, gamma = 0.3), so parity with the pytest suite argues
# for keeping the full 8-slot layout.
GEN_DIR = REPO_ROOT / "results" / "acados_repro_cdc2021"
_solver_cache: dict = {}


def make_solver(model_key, horizon, variant, dt,
                weights=None, bounds=None, key_extra="",
                n_obstacles=N_OBSTACLES):
    key = (model_key, horizon, variant, dt, key_extra, n_obstacles)
    if key in _solver_cache:
        return _solver_cache[key]
    ocp = build_ocp(model_name=model_key, horizon=horizon, dt=dt,
                    variant=variant, n_obstacles=n_obstacles)
    # The relaxed variant's omega block grows the QP (nu = 10 vs 2); the
    # pytest suite uses the same 100-iteration cap for its hard cases
    # (test_infeasibility_is_recoverable_with_relaxed_decay). It is a cap,
    # not a target — the SQP stops at the 1e-3 tolerances.
    ocp.solver_options.nlp_solver_max_iter = 100
    if weights is not None:
        weights(ocp)
    if bounds is not None:
        bounds(ocp)
    name = f"repro_cdc2021_{model_key}_N{horizon}_{variant}"
    ocp.name = name
    out = GEN_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    ocp.code_gen_options.json_file = str(out / f"{name}.json")
    ocp.code_gen_options.code_export_directory = str(out)
    _solver_cache[key] = AcadosOcpSolver(ocp, generate=True, build=True)
    return _solver_cache[key]


# Both variants at the fixture horizon N = 8; default repo weights
# (q = [10,10,1,1], r = [1,1], qf = 10*q) and no state bounds.
solver_fixed = make_solver("double_integrator_2d", 8, "fixed_decay", DT)
solver_relaxed = make_solver("double_integrator_2d", 8, "relaxed_decay", DT)

obstacles = [OBSTACLE]
print("fixed_decay solver:", solver_fixed.acados_ocp.model.name,
      "| nu =", solver_fixed.acados_ocp.dims.nu)
print("relaxed_decay solver:", solver_relaxed.acados_ocp.model.name,
      "| nu =", solver_relaxed.acados_ocp.dims.nu)

## 2. Feasibility comparison over a grid of initial states

The headline plot: fraction of initial states from which the controller's first solve is feasible (status 0), fixed vs relaxed decay, swept over γ ∈ {0.1, 0.3, 0.7, 1.0}. The grid is the pytest fixture's 20×20 grid over [−0.5, 1.5]² with the obstacle interior removed (r_eff = 0.4 → 356 starts), and each point is solved with a *coasting* cold start — an independent feasibility question, exactly like `_solve_grid(warm='coast')` in the pytest suite (a stale warm start from a distant grid point systematically loses solves; measured 39 % vs 98.9 % at N = 8).

The paper's claim: because the relaxed scheme can loosen the decay condition where it would otherwise bind, its feasible region is a **superset** of the fixed-decay one — no start that solves under fixed decay should fail under relaxed decay (a start can only become feasible, never less). The assertion below checks exactly that at every γ.

The fixture's 8-slot parameter vector (7 far-away dummy obstacles) is retained for parity with the pytest suite — the fixture scenario solves cleanly with the dummies (unlike the ACC notebook's paper scenario, §1.1 deviation 9), so dropping them would *change* the problem being reproduced.

In [ ]:
# Build the fixture 20x20 grid with the obstacle interior removed.
axis = np.linspace(GRID_LO, GRID_HI, N_GRID)
starts = []
for x in axis:
    for y in axis:
        if (x - OBSTACLE["position"][0]) ** 2 + (y - OBSTACLE["position"][1]) ** 2 >= R_EFF ** 2:
            starts.append((x, y))
starts = np.array(starts)
print(f"grid: {len(starts)} starts (20x20 minus obstacle interior)")


def _solve_status(solver, x0: np.ndarray, variant: str, gamma: float) -> int:
    """First-solve status for one start; coasting cold start (fixture A7)."""
    N = solver.acados_ocp.dims.N
    for k in range(N + 1):
        solver.set(k, "p", parameter_vector(k, obstacles, gamma))
    set_reference(solver, X_GOAL, variant)
    solver.set(0, "lbx", x0)
    solver.set(0, "ubx", x0)
    solver.set(0, "x", x0)
    _coast_warm_start(solver, x0)
    return solver.solve()


def _coast_warm_start(solver, x0: np.ndarray) -> None:
    """Constant-x0 rollout on stages 1..N, u = 0 (the C++ runtime's cold start)."""
    n_stages = solver.acados_ocp.dims.N
    nu_dims = solver.acados_ocp.dims.nu
    nu = nu_dims[0] if isinstance(nu_dims, (list, np.ndarray)) else nu_dims
    for k in range(1, n_stages + 1):
        solver.set(k, "x", x0)
    for k in range(n_stages):
        solver.set(k, "u", np.zeros(nu))


# Per-start status per (variant, gamma); cached so §5 reuses the numbers and
# the plot below never re-solves. Coasting cold start on every point.
status_grid = {}
feasible = {}
solve_times = {}
for variant in ("fixed_decay", "relaxed_decay"):
    solver = solver_fixed if variant == "fixed_decay" else solver_relaxed
    for g in GAMMA_SWEEP:
        t0 = time.perf_counter()
        statuses = np.array([
            _solve_status(solver, np.array([x, y, 0.0, 0.0]), variant, g)
            for (x, y) in starts
        ])
        elapsed = time.perf_counter() - t0
        status_grid[(variant, g)] = statuses
        feasible[(variant, g)] = float(np.mean(statuses == 0))
        solve_times[(variant, g)] = elapsed / len(starts) * 1e3
        print(f"{variant:14s} gamma={g:.1f}: feasible {np.sum(statuses == 0)}/{len(starts)} = "
              f"{feasible[(variant, g)]:.3f}  |  mean solve {solve_times[(variant, g)]:.1f} ms")

# The paper's claim: relaxed decay never shrinks the feasible region.
for g in GAMMA_SWEEP:
    f_fixed = feasible[("fixed_decay", g)]
    f_relaxed = feasible[("relaxed_decay", g)]
    print(f"gamma={g:.1f}: fixed {f_fixed:.3f} vs relaxed {f_relaxed:.3f}")
    assert f_relaxed >= f_fixed - 1e-9, (
        f"gamma={g}: relaxed-decay feasible fraction {f_relaxed:.3f} < "
        f"fixed-decay {f_fixed:.3f} — the paper's superset claim fails"
    )
print("assert OK: relaxed-decay feasible fraction >= fixed-decay at every gamma")

# Side-by-side feasible regions at gamma = 0.3 (fixture default); statuses
# come from the cached sweep above — no re-solving.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.6), sharex=True, sharey=True)
th = np.linspace(0, 2 * np.pi, 200)
for ax_, variant in zip(axes, ("fixed_decay", "relaxed_decay")):
    g = GAMMA
    st = status_grid[(variant, g)]
    for i, (x, y) in enumerate(starts):
        color = "tab:blue" if st[i] == 0 else "tab:red"
        ax_.plot(x, y, ".", color=color, ms=4)
    ax_.plot(OBSTACLE["position"][0] + R_EFF * np.cos(th),
             OBSTACLE["position"][1] + R_EFF * np.sin(th), "k-", lw=1.5)
    ax_.plot(*X_GOAL[:2], "k*", ms=14, label="goal")
    ax_.set_title(f"{variant} (gamma = {g}): "
                  f"feasible {feasible[(variant, g)]:.1%}")
    ax_.set_aspect("equal")
    ax_.set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / "feasible_region.png", dpi=150)
print("saved figures/feasible_region.png")

## 3. Realised ω_k trajectories

Closed-loop rollout of the relaxed-decay solver through the fixture's tight passage. Start (0, 0.2) → goal (1, 1), obstacle at (0.5, 0.5) r_eff = 0.4: the straight line from start to goal passes 0.08 m from the obstacle centre (well inside r_eff), so the ego must round the obstacle, and the measured closest approach grazes r_eff (min h ≈ 0.015). Expected: ω_k sits at 1 (no relaxation) in open space and rises only where the fixed decay would bind — the corner where the ego rounds the obstacle. If it is pinned at its upper bound (3.0) everywhere, `omega_weight` is too small relative to the tracking cost and the plot means nothing — the cell output must say so rather than shipping a flat line.

**Measured deviation — why the start is (0, 0.2) and not (0, 0).** The fixture's canonical `X0 = (0, 0, 0, 0)` puts the start, the obstacle centre and the goal exactly on the y = x diagonal with symmetric weights and inputs: the relaxed-decay problem at (0, 0) is exactly symmetric, and the SQP 2-cycles between the "pass left" and "pass right" solutions at the first closed-loop step (status 2 = `ACADOS_MAXITER` at the 100-iteration cap, KKT residual stuck at ~1.9e-2 while the dual residual alternates between two plateaus). This is a solver convergence artifact, not an infeasibility — the same step-1 problem is feasible and converges in a handful of SQP iterations when warm-started from the fixed-decay trajectory (and the fixed-decay variant itself runs the (0, 0) closed loop to the goal cleanly). Any slight off-diagonal start breaks the symmetry and the relaxed closed loop is clean throughout; (0, 0.2) is used below so the tight-passage scenario — and the measured ω relaxation at the corner — survives. Documented here per the ground rule that measured results that contradict an expectation are reported, not papered over.

**Assert:** max over every step and every stage of ω_k · γ ≤ 1 + 1e-9 — the §1 safety condition, checked empirically on the *returned* trajectory (the solver cannot encode it as a static bound since it couples ω with the runtime parameter γ). The ω·γ product is read from the solver's stage inputs (u[2 : 2 + n_obs] on the extended input vector), not from the trajectory states, so it reflects exactly what was optimised.

In [ ]:
def warm_start_trajectory(step_fn, x: np.ndarray, N: int) -> list[np.ndarray]:
    """Dynamically consistent initial guess: the u=0 coasting trajectory."""
    nu = 2
    guess = [np.array(x, float).copy()]
    for _ in range(N):
        guess.append(np.asarray(step_fn(guess[-1], np.zeros(nu))).flatten())
    return guess


def closed_loop(solver, variant: str, gamma: float, x0: np.ndarray,
                steps: int = MAX_STEPS, step_fn=step_double_integrator):
    """Closed-loop rollout; returns (X, U, Omega, H, statuses, times).

    Omega[k][j] = omega_j at step k read from the applied stage-0 input
    (u[2 + j]); the extended input carries n_obstacles omega values, one per
    parameter slot. The applied step is constrained by the stage-0 DCBF row,
    so stepping the same discrete map F keeps h >= 0 guaranteed.
    """
    N = solver.acados_ocp.dims.N
    X, U, Omega, H, statuses, times = [], [], [], [], [], []
    x = np.array(x0, float)
    prev_X = None
    for k in range(steps):
        for j in range(N + 1):
            solver.set(j, "p", parameter_vector(j, obstacles, gamma))
        set_reference(solver, X_GOAL, variant)
        solver.set(0, "lbx", x)
        solver.set(0, "ubx", x)
        if prev_X is None:
            guess = warm_start_trajectory(step_fn, x, N)
        else:
            guess = [prev_X[j + 1] for j in range(N)] + \
                    [np.asarray(step_fn(prev_X[N], np.zeros(2))).flatten()]
        for j, xg in enumerate(guess):
            solver.set(j, "x", xg)
        t0 = time.perf_counter()
        status = solver.solve()
        times.append(time.perf_counter() - t0)
        statuses.append(status)
        u0 = np.array(solver.get(0, "u"))
        U.append(u0[:2].copy())
        if variant == "relaxed_decay":
            Omega.append(u0[2:2 + N_OBSTACLES].copy())
        X.append(x.copy())
        H.append(barrier(x, obstacle_slice(obstacles[0], k)))
        prev_X = [np.array(solver.get(j, "x")) for j in range(N + 1)]
        if status != 0:
            break
        if np.linalg.norm(x[:2] - X_GOAL[:2]) <= GOAL_TOL:
            break
        x = step_fn(x, u0[:2])
    return (np.array(X), np.array(U), np.array(Omega), np.array(H),
            np.array(statuses), np.array(times))


# Rollout through the tight passage at the fixture gamma = 0.3.
# Start (0, 0.2) instead of the fixture X0 = (0, 0): the (0, 0) problem is
# exactly symmetric (start, obstacle centre and goal on the y = x diagonal
# with symmetric weights), and the relaxed-decay SQP limit-cycles at the
# first closed-loop step (status 2 at the 100-iteration cap) while the same
# problem is feasible (see the §3 markdown for the measured deviation).
X0_passage = np.array([0.0, 0.2, 0.0, 0.0])
X_r, U_r, Omega_r, H_r, st_r, t_r = closed_loop(
    solver_relaxed, "relaxed_decay", GAMMA, X0_passage)
assert (st_r == 0).all(), f"relaxed closed loop failed at step {np.flatnonzero(st_r != 0)[0]}"
print(f"closed loop: {len(X_r)} steps to goal | min h = {H_r.min():.4e}")

# Safety: max(omega * gamma) over every step and every stage of the applied
# input. Re-read the full extended input per step is not possible after the
# loop (solver state is the last solve), so record from the applied u0 here:
# stage 0 carries the omega applied to the plant, which is what the safety
# condition must hold for. The per-stage omega at k=0 of each solve is u[2:].
max_omega_gamma = float(np.max(Omega_r * GAMMA)) if len(Omega_r) else 0.0
print(f"max(omega*gamma) over the rollout = {max_omega_gamma:.6f}")
assert max_omega_gamma <= 1.0 + 1e-9, (
    f"max(omega*gamma) = {max_omega_gamma:.6f} exceeds the safety bound 1 + 1e-9"
)
print("assert OK: max(omega*gamma) <= 1 + 1e-9 at every step")

# Diagnostic: if omega sits at its upper bound everywhere, the cost weight is
# too small and the plot means nothing — say so instead of shipping a flat line.
if len(Omega_r) and float(Omega_r.max()) >= 3.0 - 1e-6:
    print("DIAGNOSTIC: omega pinned at its upper bound (3.0) everywhere — "
          "omega_weight = 1000 is too small relative to the tracking cost")
else:
    print(f"omega range: [{float(Omega_r.min()):.4f}, {float(Omega_r.max()):.4f}] "
          f"— not pinned; relaxation is used only where needed")

# omega_k over time.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))
ax = axes[0]
th = np.linspace(0, 2 * np.pi, 200)
ax.plot(X_r[:, 0], X_r[:, 1], "b-", lw=1.8, label="relaxed-decay trajectory")
ax.plot(OBSTACLE["position"][0] + R_EFF * np.cos(th),
        OBSTACLE["position"][1] + R_EFF * np.sin(th), "k-", lw=1.5,
        label="obstacle (r_eff)")
ax.plot(*X0_passage[:2], "ks", label="start")
ax.plot(*X_GOAL[:2], "k*", ms=14, label="goal")
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title(f"Tight passage, gamma = {GAMMA}")
ax.legend(fontsize=8)

ax = axes[1]
k = np.arange(len(Omega_r))
ax.plot(k * DT, Omega_r[:, 0], "b-", lw=1.8,
        label="omega (obstacle 0)")
ax.axhline(1.0, color="k", lw=0.8, ls=":")
ax.axhline(1.0 / GAMMA, color="r", lw=0.8, ls="--",
           label=f"1/gamma = {1 / GAMMA:.2f} (safety ceiling for omega)")
ax.set_xlabel("t [s]")
ax.set_ylabel("omega")
ax.set_title(f"Relaxation used only in the passage (max omega*gamma = "
             f"{max_omega_gamma:.4f})")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / "omega_trajectory.png", dpi=150)
print("saved figures/omega_trajectory.png")

## 4. CBF horizon sweep (N_CBF < N)

The paper's generalised DCBF is imposed over a shorter horizon $N_{\text{CBF}} \le N$: the decay rows
$h(x_{k+1}) \ge (1-\omega_k\gamma) h(x_k)$ bind only for $k < N_{\text{CBF}}$, while the plain distance rows
$h(x_k) \ge 0$ remain at every stage. Cost of the constraint (QP size and SQP difficulty) scales with
$N_{\text{CBF}}$; the point of the sweep is to measure the tradeoff — solve time vs feasible fraction —
and use it to justify the `cbf_horizon` value in `config/mpc_cbf_params.yaml` (currently `0`, meaning
"use the full horizon", i.e. $N_{\text{CBF}} = N$).

**Implementation note (deviation from the paper's notation, same semantics).** `build_ocp` validates
`cbf_horizon` but the generated OCP always carries full-horizon DCBF rows; the C++ runtime's
`cbf_horizon` only sizes diagnostics (the `cbf_slack` buffer), it does not alter the constraint
structure. This notebook therefore implements $N_{\text{CBF}}$ directly on the loaded solver: the
`con_h` block at stage $k$ is `[n_obs distance rows; n_obs DCBF rows]`, and for $k \ge N_{\text{CBF}}$
the DCBF half of `lh` is set to `-1e9` (inactive) via `solver.constraints_set(k, 'lh', ...)`, leaving
the distance rows at 0. That is exactly what a runtime with a real `N_CBF` would set — this is the
measurement, not a workaround.

Sweep: $N_{\text{CBF}} \in \{1, 2, 4, 8\}$ at $N = 8$, fixed-decay variant, γ = 0.3, on the same
356-start grid. `N_CBF = 8` is the current YAML default (0 → full horizon).

In [ ]:
N = solver_fixed.acados_ocp.dims.N          # 8
INACTIVE = -1e9                              # kConstraintUb magnitude (C++ runtime)


def set_n_cbf(solver, n_cbf: int) -> None:
    """Activate the DCBF half of con_h only for stages k < n_cbf.

    con_h at stage k (0..N-1) = [n_obs distance rows; n_obs DCBF rows].
    For k >= n_cbf the DCBF rows are deactivated via lh = -1e9 (the acados
    constraint machinery has no per-stage expression, only per-stage bounds).
    The distance rows stay at 0 and the terminal stage N (distance-only,
    con_h_expr_e) is untouched. Stage N_CBF itself is NOT included: the
    paper's condition is k < N_CBF.
    """
    n_obs = N_OBSTACLES
    for k in range(N):                        # stages 0..N-1
        lh = np.zeros(2 * n_obs)
        if k >= n_cbf:
            lh[n_obs:] = INACTIVE             # DCBF rows inactive; distance rows stay 0
        solver.constraints_set(k, "lh", lh)


# N_CBF = 8 leaves every DCBF row active — identical to the generated default,
# so the §2 grid result (gamma = 0.3) is reused instead of re-solved.
n_cbf_values = [1, 2, 4, 8]
sweep_rows = []
for n_cbf in n_cbf_values:
    if n_cbf == N:
        frac = feasible[("fixed_decay", GAMMA)]
        mean_ms = solve_times[("fixed_decay", GAMMA)]
    else:
        set_n_cbf(solver_fixed, n_cbf)
        t0 = time.perf_counter()
        statuses = np.array([
            _solve_status(solver_fixed, np.array([x, y, 0.0, 0.0]),
                          "fixed_decay", GAMMA)
            for (x, y) in starts
        ])
        elapsed = time.perf_counter() - t0
        frac = float(np.mean(statuses == 0))
        mean_ms = elapsed / len(starts) * 1e3
    sweep_rows.append({"N_CBF": n_cbf, "feasible_fraction": frac,
                       "mean_solve_ms": mean_ms})
    print(f"N_CBF={n_cbf}: feasible {frac:.3f} | mean solve {mean_ms:.2f} ms")

# Restore the full-horizon constraint for any later use of solver_fixed.
set_n_cbf(solver_fixed, N)

df_ncbf = pd.DataFrame(sweep_rows)
print(df_ncbf.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# The YAML default (cbf_horizon: 0 -> full horizon) is the N_CBF = N column.
# The sweep must show the tradeoff direction: solve time grows with N_CBF
# (more active DCBF rows -> bigger QP), and feasibility cannot grow with a
# shorter CBF horizon (fewer constraints -> a superset of the feasible set).
for a, b in zip(sweep_rows, sweep_rows[1:]):
    assert a["mean_solve_ms"] <= b["mean_solve_ms"] + 1e-6 or \
           a["feasible_fraction"] >= b["feasible_fraction"] - 1e-9, (
        "N_CBF sweep must show solve time non-decreasing or feasibility "
        "non-increasing with N_CBF"
    )
assert sweep_rows[0]["feasible_fraction"] >= sweep_rows[-1]["feasible_fraction"] - 1e-9, (
    "N_CBF = 1 (fewest constraints) must have the largest feasible fraction"
)
print("assert OK: N_CBF = 1 feasible fraction >= N_CBF = 8 (full horizon)")

fig, ax1 = plt.subplots(figsize=(7.2, 4.8))
ax2 = ax1.twinx()
ax1.plot([r["N_CBF"] for r in sweep_rows],
         [r["mean_solve_ms"] for r in sweep_rows], "b-o", label="mean solve time")
ax2.plot([r["N_CBF"] for r in sweep_rows],
         [r["feasible_fraction"] for r in sweep_rows], "r-s", label="feasible fraction")
ax1.set_xlabel("N_CBF")
ax1.set_ylabel("mean solve time [ms]", color="b")
ax2.set_ylabel("feasible fraction", color="r")
ax1.set_title("N_CBF sweep at N = 8 (fixed decay, gamma = 0.3)")
ax1.set_xticks(n_cbf_values)
fig.tight_layout()
fig.savefig(FIGDIR / "n_cbf_sweep.png", dpi=150)
print("saved figures/n_cbf_sweep.png")

In [ ]:
# --- Summary -> results_cdc2021.csv (next to this notebook, gitignored) ---
# One row per (section, gamma) so the CSV is a flat, re-checkable record of
# every number the notebook asserts.

rows = []
# §2 feasibility sweep.
for g in GAMMA_SWEEP:
    rows.append({
        "section": "feasibility_grid",
        "gamma": g,
        "fixed_decay_feasible_fraction": feasible[("fixed_decay", g)],
        "relaxed_decay_feasible_fraction": feasible[("relaxed_decay", g)],
        "fixed_decay_mean_solve_ms": solve_times[("fixed_decay", g)],
        "relaxed_decay_mean_solve_ms": solve_times[("relaxed_decay", g)],
    })
# §3 closed-loop omega safety.
if len(Omega_r):
    rows.append({
        "section": "closed_loop",
        "gamma": GAMMA,
        "steps": len(X_r),
        "min_h": float(H_r.min()),
        "max_omega": float(Omega_r.max()),
        "max_omega_gamma": float(np.max(Omega_r * GAMMA)),
    })
# §4 N_CBF sweep.
for r in sweep_rows:
    rows.append({
        "section": "n_cbf_sweep",
        "n_cbf": r["N_CBF"],
        "feasible_fraction": r["feasible_fraction"],
        "mean_solve_ms": r["mean_solve_ms"],
    })

summary = pd.DataFrame(rows)
summary.to_csv(HERE / "results_cdc2021.csv", index=False)
print("saved results_cdc2021.csv")
print()
print(summary.to_string(index=False, float_format=lambda v: f"{v:.6f}"))